In [1]:
!pip install mcp groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.7 MB/s eta 0:00:00


In [2]:
import json
import os
from google.colab import userdata
from mcp.server.fastmcp import FastMCP
from groq import Groq

# Recuperando a chave de API do painel lateral "Secrets" (ícone de chave no Colab)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("Chave API do Groq carregada com sucesso!")
except Exception as e:
    print("AVISO: Configure o secret 'GROQ_API_KEY' no menu lateral do Colab.")

# --- Bancos de Dados Simulados ---
FAQ_DB = {
    "matricula": "A matrícula ocorre entre os dias 10 e 20 de janeiro pelo portal do aluno.",
    "boleto": "Os boletos de mensalidade ficam disponíveis no portal financeiro todo dia 5.",
    "biblioteca": "A biblioteca central funciona de segunda a sexta, das 08h às 22h.",
    "dp": "As inscrições para Dependência (DP) ocorrem na primeira semana de aula."
}

CALENDARIO_DB = {
    "engenharia_civil": {"matricula": "10 a 20 de Janeiro", "provas": "15 a 30 de Junho", "tcc": "10 de Novembro"},
    "direito": {"matricula": "12 a 22 de Janeiro", "provas": "10 a 25 de Junho", "tcc": "05 de Novembro"},
    "ciencia_da_computacao": {"matricula": "10 a 20 de Janeiro", "provas": "10 a 25 de Junho", "tcc": "15 de Novembro"}
}

DOCUMENTOS_DB = {
    "2026001": {"historico": "Aprovado", "identidade": "Pendente de envio"},
    "2026002": {"historico": "Aprovado", "identidade": "Aprovado"}
}

Chave API do Groq carregada com sucesso!


In [3]:
# Instanciando o servidor MCP
mcp_server = FastMCP("FAQ_Universidade")

@mcp_server.tool()
def buscar_faq(pergunta: str) -> str:
    """Busca respostas para dúvidas frequentes na base de conhecimento."""
    pergunta_lower = pergunta.lower()
    for chave, resposta in FAQ_DB.items():
        if chave in pergunta_lower:
            return resposta
    return "Desculpe, não encontrei uma resposta automática para essa dúvida. Por favor, detalhe mais."

@mcp_server.tool()
def consultar_calendario(curso: str) -> str:
    """Retorna datas importantes (matrícula, provas) para um curso específico."""
    curso_formatado = curso.lower().replace(" ", "_")
    dados = CALENDARIO_DB.get(curso_formatado)
    if dados:
        return f"Calendário de {curso}: Matrícula: {dados['matricula']} | Provas: {dados['provas']} | TCC: {dados['tcc']}"
    return f"Curso '{curso}' não encontrado no sistema."

@mcp_server.tool()
def verificar_documento(tipo: str, ra: str) -> str:
    """Checa o status de um documento do aluno informando o tipo e o RA."""
    aluno = DOCUMENTOS_DB.get(str(ra))
    if not aluno:
        return f"Aluno com RA {ra} não encontrado."

    status = aluno.get(tipo.lower())
    if status:
        return f"O status do documento '{tipo}' para o RA {ra} é: {status}."
    return f"Documento do tipo '{tipo}' não consta nos registros para este aluno."

@mcp_server.resource("faq://todas")
def listar_faqs():
    """Retorna a lista completa de perguntas e respostas frequentes."""
    return json.dumps(FAQ_DB, ensure_ascii=False, indent=2)

print("Servidor MCP 'FAQ_Universidade' configurado com Tools e Resources!")

Servidor MCP 'FAQ_Universidade' configurado com Tools e Resources!


In [5]:
client = Groq()

# Mapeamento para invocar as funções MCP localmente no Colab
tools_map = {
    "buscar_faq": buscar_faq,
    "consultar_calendario": consultar_calendario,
    "verificar_documento": verificar_documento
}

# Definição das ferramentas para o LLM do Groq
groq_tools = [
    {
        "type": "function",
        "function": {
            "name": "buscar_faq",
            "description": "Use para responder perguntas gerais sobre regras da faculdade (ex: boletos, biblioteca, dp).",
            "parameters": {
                "type": "object",
                "properties": {"pergunta": {"type": "string", "description": "A dúvida do aluno"}},
                "required": ["pergunta"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_calendario",
            "description": "Use para consultar datas de provas e matrículas baseadas no curso.",
            "parameters": {
                "type": "object",
                "properties": {"curso": {"type": "string", "description": "Nome do curso (ex: Engenharia Civil)"}},
                "required": ["curso"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "verificar_documento",
            "description": "Use para checar o status de documentos pendentes usando o RA do aluno.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tipo": {"type": "string", "description": "Tipo de documento (ex: historico, identidade)"},
                    "ra": {"type": "string", "description": "Registro Acadêmico do aluno (ex: 2026001)"}
                },
                "required": ["tipo", "ra"],
            },
        },
    }
]

def agente_secretaria(prompt_usuario):
    print(f"\n📩 E-mail recebido: '{prompt_usuario}'")
    messages = [
        {"role": "system", "content": "Você é um assistente da secretaria da faculdade. Use as ferramentas fornecidas para ajudar os alunos."},
        {"role": "user", "content": prompt_usuario}
    ]

    # Passo 1: O Groq decide qual Tool usar
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=groq_tools,
        tool_choice="auto",
        max_tokens=500
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Passo 2: Executar a Tool no servidor MCP (simulação de chamada)
    if tool_calls:
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(f"⚙️ Agente chamando tool MCP: {function_name}({function_args})")

            # Chama a função real anotada no FastMCP
            function_response = tools_map[function_name](**function_args)

            # Adiciona o resultado na conversa
            messages.append(response_message)
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": str(function_response),
            })

        # Passo 3: O Groq gera a resposta final amigável
        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )
        print(f"🤖 Resposta do Agente:\n{final_response.choices[0].message.content}")
    else:
        print(f"🤖 Resposta do Agente:\n{response_message.content}")

# Testando os cenários
agente_secretaria("Quais são as datas de prova para Engenharia Civil?")
agente_secretaria("Meu RA é 2026001, minha identidade já foi aprovada?")
agente_secretaria("Como faço para pagar o boleto da mensalidade?")


📩 E-mail recebido: 'Quais são as datas de prova para Engenharia Civil?'
⚙️ Agente chamando tool MCP: consultar_calendario({'curso': 'Engenharia Civil'})
🤖 Resposta do Agente:
O calendário acadêmico para o curso de Engenharia Civil inclui as seguintes datas importantes:
- Período de matrícula: 10 a 20 de janeiro
- Período de provas: 15 a 30 de junho
- Apresentação do Trabalho de Conclusão de Curso (TCC): 10 de novembro

📩 E-mail recebido: 'Meu RA é 2026001, minha identidade já foi aprovada?'
⚙️ Agente chamando tool MCP: verificar_documento({'ra': '2026001', 'tipo': 'identidade'})
🤖 Resposta do Agente:
Sua identidade ainda não foi enviada para análise, portanto não há informações de aprovação ou reprovação. É necessário enviar o documento para que possamos realizar a verificação. Se tiver alguma dúvida, é só perguntar.

📩 E-mail recebido: 'Como faço para pagar o boleto da mensalidade?'
⚙️ Agente chamando tool MCP: buscar_faq({'pergunta': 'pagar boleto mensalidade'})
🤖 Resposta do Agente